# 04 · Modelado y evaluación
Lee `data/processed/dataset_final` y la ficha `ficha_candidatas.csv` (entran las variables con decisión *Entra*).

**Diseño**
* **Partición temporal:** se entrena con la(s) cohorte(s) anterior(es) y se prueba en la **última** (2025). El conjunto
  de prueba no se toca hasta el final.
* **Validación cruzada** estratificada (5 pliegues) sobre una muestra del entrenamiento; **búsqueda de
  hiperparámetros anidada** (interna: elegir; externa: estimar).
* **Métricas independientes del umbral** (PR-AUC, ROC-AUC) para comparar y elegir; **umbral operativo** elegido
  por **costo** (un falso negativo cuesta `COSTO_FN_FP` veces un falso positivo), solo con datos de entrenamiento.
* Todas las transformaciones están dentro de un `Pipeline` (ajuste solo con datos de entrenamiento).

**Salidas:** `results/comparacion_cv.csv`, `results/cifras_informe.json`, `results/cifras_informe.md`,
`models/modelo_final.joblib`, `figs/07_*.png`, `figs/08_*.png`.

In [ ]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / 'src'))
import numpy as np, pandas as pd
import proyecto as P
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
print('Raíz del proyecto:', P.RAIZ)

In [ ]:
import json, joblib
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold, cross_val_predict, cross_validate, train_test_split)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

# ------------------ SUPUESTO DE COSTO (justificar en el informe) ------------------
COSTO_FN_FP = 5     # un falso negativo (no detectar a quien fracasa) cuesta 5 veces un falso positivo (tutoría innecesaria)
COSTOS_SENSIBILIDAD = (2, 5, 10)
# -----------------------------------------------------------------------------------

ficha = pd.read_csv(P.DIR_PROC / 'ficha_candidatas.csv')
VARS = ficha.loc[ficha['decision'] == 'Entra', 'variable'].tolist()
print('Variables del modelo (según la ficha):', VARS)

df = P.leer_tabla('dataset_final')
ANIO_TEST = int(df['AGNO'].max())
train, test = df[df['AGNO'] < ANIO_TEST], df[df['AGNO'] == ANIO_TEST]
assert len(train) and len(test), 'Se necesitan al menos dos cohortes (año de entrenamiento y año de prueba)'
Xtr, ytr = train[VARS], train[P.OBJETIVO]
print(f'Entrenamiento: cohorte(s) {sorted(int(a) for a in train.AGNO.unique())} n={len(train):,} (tasa {ytr.mean()*100:.2f}%)')
print(f'Prueba (intacta hasta el final): cohorte {ANIO_TEST} n={len(test):,} (tasa {test[P.OBJETIVO].mean()*100:.2f}%)')

S = P.tamanos()
n_cv = min(S['cv'], len(train))
Xs, _, ys, _ = train_test_split(Xtr, ytr, train_size=n_cv, stratify=ytr, random_state=P.SEMILLA)
cv = StratifiedKFold(S['folds'], shuffle=True, random_state=P.SEMILLA)
CIFRAS = {'variables': VARS, 'anio_test': ANIO_TEST, 'n_train': int(len(train)), 'n_test': int(len(test)),
          'n_muestra_cv': int(n_cv), 'folds': S['folds'], 'costo_fn_fp': COSTO_FN_FP}

## 1. Comparación de modelos y baselines (validación cruzada)
Cuatro modelos de tres familias (lineal, árbol, ensambles) y dos baselines triviales. Se compara con métricas que
no dependen del umbral (PR-AUC, ROC-AUC) y, a modo de referencia, F1/Recall/Precisión con el umbral por defecto
(los modelos usan `class_weight='balanced'`).

In [ ]:
scoring = {'pr_auc': 'average_precision', 'roc_auc': 'roc_auc', 'f1': 'f1', 'recall': 'recall', 'precision': 'precision'}
resultados, crudo = [], {}
for nombre, modelo in P.modelos(VARS).items():
    r = cross_validate(modelo, Xs, ys, cv=cv, scoring=scoring, n_jobs=1)
    crudo[nombre] = {k: r[f'test_{k}'] for k in scoring}
    fila = {'modelo': nombre}
    for k in scoring:
        fila[k] = f"{r[f'test_{k}'].mean():.3f} ± {r[f'test_{k}'].std():.3f}"
    resultados.append(fila)
    print('listo:', nombre)
tabla_cv = pd.DataFrame(resultados)
tabla_cv.to_csv(P.DIR_RES / 'comparacion_cv.csv', index=False)
display(tabla_cv)
CIFRAS['comparacion_cv'] = tabla_cv.to_dict('records')

## 2. Búsqueda de hiperparámetros **anidada** (HistGradientBoosting)
* **Interna (3 pliegues):** `RandomizedSearchCV` elige hiperparámetros optimizando PR-AUC.
* **Externa (3 pliegues):** estima el desempeño del *procedimiento completo* (búsqueda incluida), sin que los datos
  que evalúan hayan participado en la elección.
Después se repite la búsqueda sobre toda la muestra de CV para fijar los hiperparámetros finales.

In [ ]:
base = P.modelos(VARS)['HistGradientBoosting']
paso = 'histgradientboostingclassifier__'
espacio = {paso + 'max_depth': [3, 4, 6, 8], paso + 'learning_rate': [0.03, 0.05, 0.1],
           paso + 'min_samples_leaf': [50, 100, 250], paso + 'max_iter': [100, 200]}
interno = StratifiedKFold(3, shuffle=True, random_state=P.SEMILLA)
externo = StratifiedKFold(3, shuffle=True, random_state=P.SEMILLA + 1)
busqueda = RandomizedSearchCV(base, espacio, n_iter=S['n_iter'], scoring='average_precision', cv=interno,
                              random_state=P.SEMILLA, n_jobs=1)

n_an = min(S['anidada'], len(Xs))
Xn, _, yn, _ = train_test_split(Xs, ys, train_size=n_an, stratify=ys, random_state=P.SEMILLA)
anidada = cross_validate(busqueda, Xn, yn, cv=externo, scoring={'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'}, n_jobs=1)
CIFRAS['cv_anidada'] = {k: [float(np.mean(anidada[f'test_{k}'])), float(np.std(anidada[f'test_{k}']))] for k in ('pr_auc', 'roc_auc')}
CIFRAS['cv_anidada']['n'] = int(n_an)
print('CV anidada  PR-AUC = %.3f ± %.3f | ROC-AUC = %.3f ± %.3f' % (*CIFRAS['cv_anidada']['pr_auc'], *CIFRAS['cv_anidada']['roc_auc']))

busqueda.fit(Xs, ys)
MEJOR = busqueda.best_params_
print('Mejores hiperparámetros:', MEJOR)
print('PR-AUC interna del mejor:', round(busqueda.best_score_, 4))
# ¿cayó algún óptimo en el borde del espacio?
for k, v in MEJOR.items():
    vals = sorted(espacio[k]); print(f"  {k.split('__')[1]} = {v}  (borde del espacio: {v in (vals[0], vals[-1])})")
CIFRAS['mejores_hiperparametros'] = {k.split('__')[1]: v for k, v in MEJOR.items()}
json.dump(CIFRAS['mejores_hiperparametros'], open(P.DIR_MODELOS / 'mejores_hiperparametros.json', 'w'), indent=2)

## 3. Contraste de las hipótesis H2 y H3 (solo validación cruzada del entrenamiento)
**H2.** ¿Agregar el nivel de enseñanza al promedio previo mejora el PR-AUC? Misma partición, mismo modelo.

In [ ]:
def cv_pr(vars_):
    m = P.modelos(vars_)['HistGradientBoosting'].set_params(**MEJOR)
    return cross_validate(m, Xs[vars_], ys, cv=cv, scoring={'pr_auc': 'average_precision'}, n_jobs=1)['test_pr_auc']

A = [v for v in ('prom_notas_norm_prev', 'prom_prev_faltante') if v in VARS]
B = A + [v for v in ('COD_ENSE',) if v in VARS]
pr_A, pr_B, pr_C = cv_pr(A), cv_pr(B), cv_pr(VARS)
dif = pr_B - pr_A
H2 = {'solo_promedio': [float(pr_A.mean()), float(pr_A.std())], 'promedio_mas_COD_ENSE': [float(pr_B.mean()), float(pr_B.std())],
      'modelo_completo': [float(pr_C.mean()), float(pr_C.std())],
      'dif_B_menos_A_media': float(dif.mean()), 'dif_B_menos_A_desv': float(dif.std()), 'pliegues_donde_B_gana': int((dif > 0).sum()), 'pliegues': int(len(dif))}
CIFRAS['H2'] = H2
print('PR-AUC  solo promedio: %.3f ± %.3f | + COD_ENSE: %.3f ± %.3f | completo: %.3f ± %.3f' % (*H2['solo_promedio'], *H2['promedio_mas_COD_ENSE'], *H2['modelo_completo']))
print('Diferencia (B−A): %.4f ± %.4f; B gana en %d de %d pliegues' % (H2['dif_B_menos_A_media'], H2['dif_B_menos_A_desv'], H2['pliegues_donde_B_gana'], H2['pliegues']))

**H3.** Tratamiento de ausencias: *(propuesto)* faltante con bandera frente a *(ingenuo)* nota faltante = 0,0, lo que
marca `riesgo_academico_prev = 1`. Se compara la tasa de falsos positivos **entre estudiantes sin nota previa**, con
la misma cantidad total de alertas (15 %) en ambas versiones.

In [ ]:
TASA_ALERTAS = 0.15
def oof(vars_, Xdata):
    m = P.modelos(vars_)['HistGradientBoosting']
    return cross_val_predict(m, Xdata[vars_], ys, cv=cv, method='predict_proba', n_jobs=1)[:, 1]

p_prop = oof(VARS, Xs)
Xb = Xs.copy()
Xb['prom_notas_norm_prev'] = Xb['prom_notas_norm_prev'].fillna(0.0)
Xb['riesgo_academico_prev'] = (Xb['prom_notas_norm_prev'] < 4.0).astype('int8')
VARS_B = [v for v in VARS if v != 'prom_prev_faltante']
p_ing = oof(VARS_B, Xb)

sin_nota = (Xs['prom_prev_faltante'] == 1).values if 'prom_prev_faltante' in Xs else np.zeros(len(Xs), bool)
def fpr_sub(p):
    t = np.quantile(p, 1 - TASA_ALERTAS)
    pred = p >= t
    neg = (ys.values == 0) & sin_nota
    return float((pred & neg).sum() / max(1, neg.sum())), float(average_precision_score(ys, p))
f_prop, ap_prop = fpr_sub(p_prop); f_ing, ap_ing = fpr_sub(p_ing)
CIFRAS['H3'] = {'n_sin_nota_previa_en_muestra': int(sin_nota.sum()), 'tasa_alertas_fija': TASA_ALERTAS,
                'FPR_sin_nota_propuesto_bandera': f_prop, 'FPR_sin_nota_ingenuo_cero_igual_riesgo': f_ing,
                'PR_AUC_propuesto': ap_prop, 'PR_AUC_ingenuo': ap_ing}
print(json.dumps(CIFRAS['H3'], indent=2))

## 4. Umbral operativo por **costo**
Con las predicciones fuera de muestra (*out-of-fold*) del entrenamiento se elige el umbral que minimiza
`COSTO_FN_FP · FN + FP`. El umbral **no** se calibra con el conjunto de prueba.
Se muestra la sensibilidad al supuesto de costo.

In [ ]:
mejor_pipe = clone(base).set_params(**MEJOR)
p_oof = cross_val_predict(mejor_pipe, Xs, ys, cv=cv, method='predict_proba', n_jobs=1)[:, 1]
sens = []
for r in COSTOS_SENSIBILIDAD:
    t, c = P.umbral_por_costo(ys, p_oof, r)
    m = P.metricas_en_umbral(ys, p_oof, t)
    sens.append({'razon_FN_FP': r, 'umbral': round(t, 3), 'recall': round(m['recall'], 3), 'precision': round(m['precision'], 3),
                 'f1': round(m['f1'], 3), 'pct_estudiantes_con_alerta': round(m['tasa_alertas'] * 100, 1)})
display(pd.DataFrame(sens))
UMBRAL, _ = P.umbral_por_costo(ys, p_oof, COSTO_FN_FP)
CIFRAS['umbral_por_costo'] = float(UMBRAL); CIFRAS['sensibilidad_costo'] = sens
print('Umbral elegido (costo FN/FP = %s): %.3f' % (COSTO_FN_FP, UMBRAL))

## 5. Modelo final y evaluación **única** en prueba
El modelo se reentrena con **todo** el conjunto de entrenamiento y se evalúa **una sola vez** sobre la cohorte de
prueba. Se reportan el umbral por defecto (0,5) y el umbral por costo. También los baselines.

In [ ]:
final = clone(base).set_params(**MEJOR).fit(Xtr, ytr)
Xte, yte = test[VARS], test[P.OBJETIVO].values
p_te = final.predict_proba(Xte)[:, 1]

prev = float(yte.mean())
res = {'prevalencia_test': prev, 'pr_auc': float(average_precision_score(yte, p_te)), 'roc_auc': float(roc_auc_score(yte, p_te)),
       'en_umbral_0.5': P.metricas_en_umbral(yte, p_te, 0.5), 'en_umbral_costo': P.metricas_en_umbral(yte, p_te, UMBRAL),
       'top_k': [P.recall_precision_top_k(yte, p_te, k) for k in (0.05, 0.10, 0.20)]}
base_res = {}
for nombre in ('Baseline (clase mayoritaria)', 'Baseline (aleatorio estratificado)'):
    b = P.modelos(VARS)[nombre].fit(Xtr, ytr)
    pb = b.predict_proba(Xte)[:, 1]
    pred = b.predict(Xte)
    mb = P.metricas_en_umbral(yte, pred.astype(float), 0.5)
    base_res[nombre] = {'f1': mb['f1'], 'recall': mb['recall'], 'precision': mb['precision'], 'pr_auc': float(average_precision_score(yte, pb))}
res['baselines'] = base_res
CIFRAS['test'] = res
for k in ('en_umbral_0.5', 'en_umbral_costo'):
    m = res[k]; print(f"{k}: umbral={m['umbral']:.3f} recall={m['recall']:.3f} precisión={m['precision']:.3f} F1={m['f1']:.3f} | TP={m['TP']:,} FP={m['FP']:,} FN={m['FN']:,} TN={m['TN']:,}")
print(f"PR-AUC={res['pr_auc']:.3f} (prevalencia {prev:.3f}) | ROC-AUC={res['roc_auc']:.3f}")
print('Baselines:', json.dumps(base_res, indent=1))

### 5.1 Matriz de confusión, falsos negativos y alertas limitadas
El porcentaje de falsos negativos **se calcula** (FN / positivos reales) y se examina quiénes son.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, k, tit in zip(axes, ('en_umbral_0.5', 'en_umbral_costo'), ('Umbral 0,5', f'Umbral por costo ({UMBRAL:.2f})')):
    m = res[k]; mat = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    ax.imshow(mat, cmap='Blues')
    for (i, j), v in np.ndenumerate(mat): ax.text(j, i, f'{v:,}', ha='center', va='center', color='black')
    ax.set_xticks([0, 1], ['Pred 0', 'Pred 1']); ax.set_yticks([0, 1], ['Real 0', 'Real 1']); ax.set_title(tit)
fig.savefig(P.DIR_FIGS / '07_matriz_confusion.png', bbox_inches='tight'); plt.show()

m = res['en_umbral_costo']
pred = p_te >= UMBRAL
fn_mask, tp_mask = (~pred) & (yte == 1), pred & (yte == 1)
pct_fn = m['FN'] / max(1, m['FN'] + m['TP']) * 100
res['pct_falsos_negativos_entre_positivos'] = float(pct_fn)
res['fn_sin_riesgo_previo_pct'] = float((test['riesgo_academico_prev'].values[fn_mask] == 0).mean() * 100) if fn_mask.any() else None
res['tp_sin_riesgo_previo_pct'] = float((test['riesgo_academico_prev'].values[tp_mask] == 0).mean() * 100) if tp_mask.any() else None
print('Falsos negativos (umbral por costo): %.1f%% de los reprobados/retirados reales' % pct_fn)
print('  FN sin riesgo académico previo: %s%% | TP sin riesgo académico previo: %s%%' % (res['fn_sin_riesgo_previo_pct'], res['tp_sin_riesgo_previo_pct']))
print('Si solo se pudiera alertar al 10%% de los estudiantes: recall = %.3f, precisión = %.3f' % (res['top_k'][1]['recall'], res['top_k'][1]['precision']))

pr, rc, _ = precision_recall_curve(yte, p_te)
fig, ax = plt.subplots(figsize=(5, 4)); ax.plot(rc, pr, color='#c0504d'); ax.axhline(prev, ls='--', color='gray', label='prevalencia')
ax.set_xlabel('Recall'); ax.set_ylabel('Precisión'); ax.legend(); ax.set_title('Curva precisión-recall (prueba)')
fig.savefig(P.DIR_FIGS / '08_curva_pr.png', bbox_inches='tight'); plt.show()

## 6. Guardado del modelo y de las cifras del informe

In [ ]:
joblib.dump(final, P.DIR_MODELOS / 'modelo_final.joblib')
P.guardar_json(CIFRAS, 'cifras_informe.json')

t = res['en_umbral_costo']; t5 = res['en_umbral_0.5']
md_txt = f'''# Cifras para el informe (generadas por 04-modelado.ipynb)
- Entrenamiento: cohorte(s) {sorted(int(a) for a in train.AGNO.unique())}, n = {len(train):,}. Prueba: cohorte {ANIO_TEST}, n = {len(test):,} (prevalencia {prev*100:.2f}%).
- Variables: {VARS}
- Hiperparámetros elegidos: {CIFRAS['mejores_hiperparametros']}
- CV anidada (n={CIFRAS['cv_anidada']['n']:,}): PR-AUC {CIFRAS['cv_anidada']['pr_auc'][0]:.3f} ± {CIFRAS['cv_anidada']['pr_auc'][1]:.3f}
- Prueba, umbral por costo ({UMBRAL:.3f}, costo FN/FP = {COSTO_FN_FP}): recall {t['recall']:.3f}, precisión {t['precision']:.3f}, F1 {t['f1']:.3f}; TP={t['TP']:,} FP={t['FP']:,} FN={t['FN']:,} TN={t['TN']:,}
- Prueba, umbral 0,5: recall {t5['recall']:.3f}, precisión {t5['precision']:.3f}, F1 {t5['f1']:.3f}
- PR-AUC {res['pr_auc']:.3f} | ROC-AUC {res['roc_auc']:.3f}
- Falsos negativos: {res['pct_falsos_negativos_entre_positivos']:.1f}% de los positivos reales
'''
(P.DIR_RES / 'cifras_informe.md').write_text('\n'.join(l.strip() for l in md_txt.splitlines()), encoding='utf-8')
print((P.DIR_RES / 'cifras_informe.md').read_text())